# HeartShare harmonization — BDC runtime notebook

This is the primary, thin BDC notebook. It loads the versioned release
manifest, verifies the declared HeartShare runtime archive, installs bundled
offline wheels when needed, and then runs the shared harmonization package.

Edit Paths, Studies, Variables, and the Step 5 controls. The checked-in
standalone notebook remains available when a release does not yet carry a
runtime bundle.


## 1. Paths


In [ ]:
# ---------------------------------------------------------------------------
# 1. PATHS — edit these to match your project.
# ---------------------------------------------------------------------------
from datetime import datetime, timezone
from pathlib import Path

# The release bundle: catalog + standards library + mapping files.
# On BDC this is the folder that contains the versioned release directories.
RELEASE_ROOT = "/sbgenomics/project-files/x01_harmonization"
RELEASE_VERSION = "latest"          # or a specific folder, e.g. "v0.0.1"

# Advanced override only: normally leave this empty and use the versioned
# catalog paths. Add a staged folder only when its catalog path is unavailable.
# A folder here changes source-file resolution, not the STUDIES selection.
STUDY_DATA_DIRS = []

# Where results go. None means /sbgenomics/workspace/output-files/harmonized_<date>_<run_id>,
# which syncs back to the Seven Bridges project automatically.
OUTPUT_DIR = None

# Names this run in the output folder and in every output row's provenance.
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

# Read only the first N participants per file. Use a small number for a first
# smoke test, then set it back to None for the real pull.
LIMIT = None

# "warn" records unexpected categorical codes and continues.
# "strict" fails the affected dataset instead. Use it to verify a clean run.
DRIFT_POLICY = "warn"

# The notebook shows warnings and errors. Detailed INFO messages are written to
# pipeline.log in the run folder.
LOG_LEVEL = "WARNING"

print(f"Release : {RELEASE_ROOT} ({RELEASE_VERSION})")
if STUDY_DATA_DIRS:
    print(f"Sources : {len(STUDY_DATA_DIRS)} staged folder override(s)")
else:
    print("Sources : release catalog paths (no staged folder overrides)")
print(f"Run ID  : {RUN_ID}")
print("Mode    : choose DRY_RUN and BASELINE_ONLY in Step 5")


In [ ]:
"""Small standard-library bootstrap used by HeartShare thin BDC notebooks."""

from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path
from zipfile import ZipFile


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def runtime_spec_from_release(manifest_path: Path) -> dict[str, object]:
    try:
        import yaml
    except ImportError as exc:
        raise RuntimeError("PyYAML is required to read the release manifest") from exc
    document = yaml.safe_load(Path(manifest_path).read_text(encoding="utf-8")) or {}
    spec = document.get("runtime_bundle")
    if not isinstance(spec, dict) or not spec.get("path") or not spec.get("sha256"):
        raise ValueError(
            f"Release manifest has no complete runtime_bundle path/checksum: {manifest_path}"
        )
    resolved = dict(spec)
    path = Path(str(spec["path"]))
    if not path.is_absolute():
        path = (Path(manifest_path).parent / path).resolve()
    resolved["path"] = path
    return resolved


def _safe_extract(archive_path: Path, target: Path) -> None:
    with ZipFile(archive_path) as archive:
        root = target.resolve()
        for member in archive.infolist():
            destination = (target / member.filename).resolve()
            if destination != root and root not in destination.parents:
                raise ValueError(f"Unsafe path in runtime archive: {member.filename}")
        archive.extractall(target)


def _verify_extracted(target: Path) -> dict[str, object]:
    manifest_path = target / "runtime_manifest.json"
    if not manifest_path.exists():
        raise ValueError("Runtime archive is missing runtime_manifest.json")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for name, expected in (manifest.get("files") or {}).items():
        path = target / name
        if not path.is_file():
            raise ValueError(f"Runtime archive is missing declared file: {name}")
        actual = sha256_file(path)
        if actual != expected:
            raise ValueError(
                f"Runtime file checksum mismatch for {name}: expected {expected}, got {actual}"
            )
    return manifest


def _install_offline_wheels(target: Path, manifest: dict[str, object]) -> Path | None:
    requirements = list(manifest.get("required_offline_distributions") or [])
    if not requirements:
        return None
    wheelhouse = target / "wheelhouse"
    vendor = target / "vendor"
    marker = vendor / ".heartshare_wheels_installed"
    if marker.exists():
        return vendor
    vendor.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-index",
        "--find-links",
        str(wheelhouse),
        "--target",
        str(vendor),
        *requirements,
    ]
    subprocess.run(command, check=True)
    marker.write_text("ok\n", encoding="utf-8")
    return vendor


def bootstrap_runtime(
    archive_path: Path,
    expected_sha256: str,
    *,
    workspace_root: Path | None = None,
    expected_output_schema: str = "1.1",
) -> tuple[Path, dict[str, object]]:
    archive_path = Path(archive_path)
    if not archive_path.is_file():
        raise FileNotFoundError(f"Runtime bundle not found: {archive_path}")
    actual_sha256 = sha256_file(archive_path)
    if actual_sha256 != expected_sha256:
        raise ValueError(
            "Runtime bundle checksum mismatch: "
            f"expected {expected_sha256}, got {actual_sha256}"
        )

    workspace_root = Path(
        workspace_root
        or os.environ.get(
            "HEARTSHARE_VENDOR_ROOT", "/sbgenomics/workspace/_heartshare_vendor"
        )
    )
    workspace_root.mkdir(parents=True, exist_ok=True)
    target = workspace_root / actual_sha256[:16]
    if not (target / "runtime_manifest.json").exists():
        with tempfile.TemporaryDirectory(
            prefix="heartshare-runtime-", dir=workspace_root
        ) as temporary:
            staging = Path(temporary) / "unpacked"
            staging.mkdir()
            _safe_extract(archive_path, staging)
            _verify_extracted(staging)
            try:
                staging.rename(target)
            except FileExistsError:
                shutil.rmtree(staging)

    manifest = _verify_extracted(target)
    if str(manifest.get("output_schema_version")) != expected_output_schema:
        raise ValueError(
            "Runtime/output schema mismatch: "
            f"expected {expected_output_schema}, runtime provides "
            f"{manifest.get('output_schema_version')}"
        )

    vendor = _install_offline_wheels(target, manifest)
    import_root = target / str(manifest.get("import_root") or "bdc/src")
    for path in [vendor, target, import_root]:
        if path is not None and str(path) not in sys.path:
            sys.path.insert(0, str(path))

    os.environ["HEARTSHARE_RUNTIME_BUNDLE_PATH"] = str(archive_path)
    os.environ["HEARTSHARE_RUNTIME_BUNDLE_SHA256"] = actual_sha256
    os.environ["HEARTSHARE_RUNTIME_VERSION"] = str(
        manifest.get("runtime_version") or ""
    )
    os.environ["HEARTSHARE_RUNTIME_BUILD_GIT_SHA"] = str(
        manifest.get("build_git_sha") or ""
    )
    os.environ["HEARTSHARE_RUNTIME_BUILD_GIT_BRANCH"] = str(
        manifest.get("build_git_branch") or "runtime-bundle"
    )
    os.environ["HEARTSHARE_RUNTIME_BUILD_GIT_CLEAN"] = str(
        bool(manifest.get("build_git_clean", True))
    ).lower()
    return import_root, manifest


# Resolve the release manifest without importing the runtime it declares.
_release_root = Path(RELEASE_ROOT)
_release_version = str(RELEASE_VERSION)
if _release_version == "latest":
    import re
    _versions = []
    for _child in _release_root.iterdir():
        _match = re.fullmatch(r"v(\d+)\.(\d+)\.(\d+)", _child.name)
        if _child.is_dir() and _match:
            _versions.append((tuple(int(_part) for _part in _match.groups()), _child.name))
    if not _versions:
        raise FileNotFoundError(f"No versioned release folders under {_release_root}")
    _release_version = sorted(_versions)[-1][1]
_release_manifest_path = (
    _release_root / _release_version / f"release_manifest_{_release_version}.yaml"
)
_runtime_spec = runtime_spec_from_release(_release_manifest_path)
_runtime_root, _runtime_manifest = bootstrap_runtime(
    Path(_runtime_spec["path"]),
    str(_runtime_spec["sha256"]),
    expected_output_schema="1.1",
)
from heartshare_harmonization import pull_and_harmonize as pipeline
print(
    f"Runtime {_runtime_manifest['runtime_version']} ready from "
    f"{_runtime_spec['path']}"
)


## 2. What can I pick?


In [ ]:
# ---------------------------------------------------------------------------
# 2. WHAT CAN I PICK? — read-only. Lists the study and variable names that the
#    release bundle actually offers, so the next two cells are copy-paste.
# ---------------------------------------------------------------------------
from collections import defaultdict

from heartshare_harmonization.catalog import load_catalog
from heartshare_harmonization.pull_and_harmonize import load_standards_library
from heartshare_harmonization.release import (
    load_release_manifest,
    resolve_release_manifest_path,
)

_manifest_path = resolve_release_manifest_path(Path(RELEASE_ROOT), RELEASE_VERSION)
_release = load_release_manifest(_manifest_path)
_catalog = load_catalog(_release.catalog)
_standards = load_standards_library(_release.standards_library)

print(f"Release {_release.release_version}  ({_manifest_path})\n")

print("STUDIES — use these names in the STUDIES cell")
print("-" * 60)
for _name in _catalog.study_names():
    _study = _catalog.study(_name)
    _n = len(_study.datasets)
    _mapped = "mapped" if _study.hdo_mapping_file else "NO MAPPING FILE"
    print(f"  {_name:<20} {_n} file(s)   {_mapped}")

_by_category = defaultdict(list)
for _key, _definition in _standards.items():
    _by_category[_definition.get("category") or "other"].append(_key)

print("\nVARIABLES — use these names in the VARIABLES cell")
print("   (the category headings are the values for DOMAINS)")
print("-" * 60)
for _category in sorted(_by_category):
    print(f"\n  [{_category}]")
    for _line_start in range(0, len(sorted(_by_category[_category])), 4):
        _chunk = sorted(_by_category[_category])[_line_start:_line_start + 4]
        print("    " + "".join(f"{_v:<24}" for _v in _chunk).rstrip())

print(f"\n{len(_standards)} variables across {len(_by_category)} domains.")


## 3. Studies


In [ ]:
# ---------------------------------------------------------------------------
# 3. STUDIES — which studies to harmonize.
#    Leave the list empty to explicitly select every study in the release
#    catalog. The dry-run readiness check verifies every selected source path.
# ---------------------------------------------------------------------------
STUDIES = [
    "HFN-NEAT",
    # "TOPCAT",
]

_selected_studies_preview = list(STUDIES) if STUDIES else _catalog.study_names()
print(f"Studies ({len(_selected_studies_preview)}): {_selected_studies_preview}")


## 4. Variables


In [ ]:
# ---------------------------------------------------------------------------
# 4. VARIABLES — which harmonized variables you want.
#
#    VARIABLES wins if it is non-empty.
#    Otherwise DOMAINS is used.
#    If both are empty you get every mapped variable.
# ---------------------------------------------------------------------------
VARIABLES = [
    "age",
    "sex",
    "bmi",
]

DOMAINS = [
    # "demographics",
    # "anthropometric",
    # "vitals",
]

if VARIABLES:
    print(f"Selecting {len(VARIABLES)} variable(s) by name: {VARIABLES}")
elif DOMAINS:
    print(f"Selecting every variable in domain(s): {DOMAINS}")
else:
    print("Selecting ALL mapped variables.")


## 5. Run


In [ ]:
# ---------------------------------------------------------------------------
# 5. RUN — builds the command line and calls the pipeline in this kernel.
#    Edit these controls, then rerun this cell without returning to Step 1.
# ---------------------------------------------------------------------------
import sys

# Start with a dry run. Change to False here for the real pull.
DRY_RUN = True

# True keeps only mapping-designated baseline visits (including documented
# compatibility aliases such as a_base). False includes selected/all visits.
BASELINE_ONLY = True

# Repeated dependency values are never silently averaged. Add an explicitly
# reviewed rule only when needed, e.g. "TOPCAT:weight:median".
DERIVATION_REDUCERS = []

_selected_studies = list(STUDIES) if STUDIES else _catalog.study_names()
_unknown_studies = sorted(set(_selected_studies) - set(_catalog.study_names()))
if _unknown_studies:
    raise ValueError(
        f"Study name(s) not present in release {_release.release_version}: "
        f"{_unknown_studies}"
    )

argv = [
    "heartshare-harmonize",
    "--release-root", str(RELEASE_ROOT),
    "--release-version", str(RELEASE_VERSION),
    "--run-id", str(RUN_ID),
    "--categorical-drift-policy", str(DRIFT_POLICY),
    "--log-level", str(LOG_LEVEL),
    # Always on: the CSV is the deliverable, and a study without a mapping file
    # should be skipped with a warning rather than aborting the whole run.
    "--emit-csv",
    "--skip-missing-mappings",
    # Per-participant HDO JSON emission is not implemented in this writer; it
    # would only write an apology into the manifest.
    "--no-emit-hdos",
]

if OUTPUT_DIR:
    argv += ["--output-dir", str(OUTPUT_DIR)]
if STUDY_DATA_DIRS:
    argv += ["--study-data-dirs", *[str(p) for p in STUDY_DATA_DIRS]]
# Always pass the resolved selection. If omitted, the CLI infers studies from
# STUDY_DATA_DIRS and can silently narrow an empty STUDIES selection.
argv += ["--studies", *_selected_studies]
if VARIABLES:
    argv += ["--variables", *VARIABLES]
elif DOMAINS:
    argv += ["--domains", *DOMAINS]
if LIMIT:
    argv += ["--limit", str(LIMIT)]
if BASELINE_ONLY:
    argv += ["--baseline-only"]
if DERIVATION_REDUCERS:
    argv += ["--derivation-reducers", *DERIVATION_REDUCERS]
if DRY_RUN:
    argv += ["--dry-run"]

# Resolve the same output folder the pipeline will choose, so the next cell
# reads the run that just happened rather than guessing at the path.
RUN_DIR = pipeline._resolve_output_dir(
    Path(OUTPUT_DIR) if OUTPUT_DIR else None, str(RUN_ID), None
)

print("Command:")
print("  " + " ".join(argv[1:]))
print(f"\nOutput folder: {RUN_DIR}\n" + "-" * 70)

_saved_argv = sys.argv
try:
    sys.argv = argv
    EXIT_CODE = pipeline.main()
finally:
    sys.argv = _saved_argv

print("-" * 70)
if EXIT_CODE == 0:
    print(f"OK. Wrote:")
    for _f in sorted(Path(RUN_DIR).iterdir()):
        print(f"  {_f.name:<28} {_f.stat().st_size:>10,} bytes")
else:
    print(f"FAILED (exit code {EXIT_CODE}). Read the log above, then check")
    print(f"  {Path(RUN_DIR) / 'manifest.json'}  ->  the 'error' key.")
    raise RuntimeError(
        f"Harmonization failed with exit code {EXIT_CODE}; output inspection was stopped."
    )


## 6. Look at the output


In [ ]:
# ---------------------------------------------------------------------------
# 6. LOOK AT THE OUTPUT — what did we actually get?
# ---------------------------------------------------------------------------
import json

import pandas as pd

try:
    from IPython.display import display
except ImportError:  # plain python, e.g. the automated notebook test
    display = print

if DRY_RUN:
    _plan = json.loads((Path(RUN_DIR) / "dry_run_plan.json").read_text())
    manifest = json.loads((Path(RUN_DIR) / "manifest.json").read_text())
    _plan_frame = pd.DataFrame(_plan)
    _display_columns = [
        "study", "dataset_id", "exam_label", "status",
        "mount_exists", "mapping_file_exists", "notes",
    ]
    if _plan_frame.empty:
        print("NOT READY — the selected studies produced no datasets.")
    else:
        _source_files_found = int(_plan_frame["mount_exists"].sum())
        _mapping_files_found = int(_plan_frame["mapping_file_exists"].sum())
        print(
            f"DRY RUN — {_plan_frame['study'].nunique()} study/studies selected\n"
            f"{len(_plan_frame)} datasets planned\n"
            f"{_source_files_found} source files found\n"
            f"{_mapping_files_found} dataset mapping checks passed\n"
            "No data was read."
        )

    _not_ready = _plan_frame[
        (_plan_frame["status"] != "planned")
        | (~_plan_frame["mount_exists"])
        | (~_plan_frame["mapping_file_exists"])
    ] if not _plan_frame.empty else _plan_frame
    if _not_ready.empty:
        print("\nREADY — every planned source file and mapping exists.")
        print("Set DRY_RUN = False in Step 5 and re-run this cell.")
    else:
        print(f"\nNOT READY — {len(_not_ready)} dataset(s) need attention.")
        display(_not_ready[_display_columns])
        print("Fix the paths or mappings shown above before setting DRY_RUN = False.")

    _coverage = pd.DataFrame(manifest.get("variable_coverage") or [])
    if not _coverage.empty:
        print("\nPLANNED VARIABLE COVERAGE")
        display(_coverage[[
            "study", "standard_name", "timepoint", "status", "detail"
        ]])
else:
    long_df = pd.read_csv(Path(RUN_DIR) / "harmonized_long.csv")
    manifest = json.loads((Path(RUN_DIR) / "manifest.json").read_text())

    print(f"{len(long_df):,} rows x {len(long_df.columns)} columns")
    print(f"{long_df['participant_id'].nunique():,} participants, "
          f"{long_df['standard_name'].nunique()} variables, "
          f"{long_df['study'].nunique()} studies\n")

    print("BY STUDY")
    print(long_df.groupby("study").agg(
        rows=("standard_name", "size"),
        participants=("participant_id", "nunique"),
        variables=("standard_name", "nunique"),
    ).to_string())

    print("\nBY VARIABLE")
    print(long_df.groupby(["study", "standard_name"]).size().to_string())

    if "timepoint" in long_df.columns:
        _timepoints = sorted(long_df["timepoint"].dropna().astype(str).unique())
        if len(_timepoints) > 1:
            print("\nBY STUDY AND TIMEPOINT")
            print(
                long_df.groupby(["study", "timepoint"], dropna=False).agg(
                    rows=("standard_name", "size"),
                    participants=("participant_id", "nunique"),
                    variables=("standard_name", "nunique"),
                ).to_string()
            )

    _coverage = pd.DataFrame(manifest.get("variable_coverage") or [])
    if not _coverage.empty:
        print("\nVARIABLE COVERAGE")
        print(
            _coverage.groupby("status").size().rename("study-variable-timepoints")
            .to_string()
        )
        _attention = _coverage[~_coverage["status"].isin(["produced", "planned"])]
        if not _attention.empty:
            display(_attention[[
                "study", "standard_name", "timepoint", "status", "detail"
            ]].head(30))

    _derivations = pd.DataFrame(manifest.get("cross_dataset_derivations") or [])
    if not _derivations.empty:
        print("\nCROSS-DATASET DERIVATIONS")
        _derivation_columns = [
            name for name in [
                "study", "standard_name", "timepoint", "derived",
                "measured_retained", "missing_height", "missing_weight",
                "ambiguous_height", "ambiguous_weight",
                "material_disagreements",
            ]
            if name in _derivations.columns
        ]
        display(_derivations[_derivation_columns])

    # This list is derived from study/standard/timepoint coverage. Physical
    # datasets that do not contain a requested variable are not counted when
    # another selected dataset produced that variable successfully.
    _missing = manifest.get("missing_variables") or []
    print(f"\nMISSING REQUESTED VARIABLES OR VALUES: {len(_missing)}")
    for _record in _missing[:20]:
        _location = _record.get("study")
        if _record.get("dataset_id"):
            _location += f"/{_record['dataset_id']}"
        print(f"  {_location}: {_record.get('standard_name')} — "
              f"{_record.get('reason')}")

    # Source codes that were not in the mapping's value_map. These are silently
    # nulled under the "warn" policy, so they are worth reading every run.
    _drift = [
        _d for _d in (manifest.get("categorical_recode_diagnostics") or [])
        if _d.get("unexpected_value_examples")
    ]
    print(f"\nCATEGORICAL RECODE CHECK: {len(_drift)} unmapped source-code group(s)")
    if not _drift:
        print("  ✓ Every observed categorical source code was covered by its mapping.")
    for _d in _drift[:20]:
        _examples = ", ".join(str(_e.get("value")) for _e in _d["unexpected_value_examples"])
        print(f"  {_d.get('study')}/{_d.get('standard_name')} "
              f"({_d.get('source_variable')}): {_examples}")

    print("\nFIRST 10 HARMONIZED VALUES")
    _preview = long_df.copy()
    _preview["visit"] = _preview["timepoint"].fillna("not assigned")
    display(_preview[[
        "participant_id", "study", "visit", "standard_name", "value", "units"
    ]].head(10))
    print(
        "Categorical labels are stored in value; canonical category codes, when "
        "defined, are stored in value_numeric. Full source provenance is in "
        "harmonized_lineage.parquet."
    )
